In [2]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import logging
from sklearn.multioutput import MultiOutputClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
import networkx as nx

import logging
from typing import List, Tuple, Dict, Any

import networkx as nx
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.multioutput import ClassifierChain
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

# Konfiguracja logowania
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

In [3]:
import logging
from typing import List, Any, Union

import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem.MolStandardize import rdMolStandardize
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from skfp.fingerprints import ECFPFingerprint

# Konfiguracja logowania do śledzenia przepływu
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")


class MolecularStandardizer(BaseEstimator, TransformerMixin):
    """Transformator scikit-learn do parsowania i standaryzacji struktur molekularnych.

    Klasa hermetyzuje operacje RDKit. Konwertuje ciągi SMILES na obiekty grafowe,
    a następnie przeprowadza proces odsalania (wybór największego fragmentu)
    oraz neutralizacji ładunków, co jest krytyczne dla czystości wektorów cech.
    """

    def __init__(self) -> None:
        """Inicjalizuje moduły standaryzujące z biblioteki RDKit."""
        self.fragment_chooser = rdMolStandardize.LargestFragmentChooser()
        self.uncharger = rdMolStandardize.Uncharger()

    def fit(self, X: List[str], y: Any = None) -> "MolecularStandardizer":
        """Metoda fit dla kompatybilności z interfejsem scikit-learn.

        Args:
            X: Lista wejściowych ciągów SMILES.
            y: Etykiety (ignorowane).

        Returns:
            Instancja samego siebie.
        """
        return self

    def transform(self, X: List[str]) -> List[Chem.Mol]:
        """Konwertuje i standaryzuje listę ciągów SMILES.

        Args:
            X: Lista ciągów SMILES.

        Returns:
            List[Chem.Mol]: Lista zstandaryzowanych obiektów RDKit Mol.

        Raises:
            ValueError: W przypadku błędu parsowania, uniemożliwiającego
            dalsze generowanie cech dla danej próbki.
        """
        standardized_mols = []
        for smiles in X:
            mol = Chem.MolFromSmiles(str(smiles))
            if mol is None:
                raise ValueError(
                    f"Krytyczny błąd parsowania SMILES: '{smiles}'. "
                    f"Zbiór danych musi być oczyszczony przed rurociągiem."
                )

            try:
                # Odsalanie (usunięcie np. jonów [Na+], [Cl-])
                mol = self.fragment_chooser.choose(mol)
                # Neutralizacja ładunków dla spójnej reprezentacji
                mol = self.uncharger.uncharge(mol)
            except Exception as e:
                logging.debug(f"Błąd standaryzacji dla {smiles}: {e}. Używam oryginału.")

            standardized_mols.append(mol)

        return standardized_mols


def build_feature_pipeline(fp_size: int = 2048, radius: int = 2) -> Pipeline:
    """Buduje zintegrowany rurociąg do generowania cech chemicznych.

    Wykorzystuje zliczeniową wersję Extended-Connectivity Fingerprints (ECFP),
    która zachowuje informacje o wielokrotności występowania podstruktur.

    Args:
        fp_size: Wymiarowość wektora cech (liczba wygenerowanych kolumn).
        radius: Promień poszukiwań podstruktur (radius=2 odpowiada ECFP4).

    Returns:
        Pipeline: Skonfigurowany rurociąg scikit-learn.
    """
    return Pipeline([
        ("standardizer", MolecularStandardizer()),
        ("feature_extractor", ECFPFingerprint(
            radius=radius,
            fp_size=fp_size,
            count=True,
            n_jobs=-1
        ))
    ])


def create_engineered_dataset(df: pd.DataFrame, smiles_col: str = "SMILES", fp_size: int = 2048) -> pd.DataFrame:
    """Uruchamia rurociąg i scala oryginalne dane z wygenerowanymi cechami.

    Args:
        df: Bazowa ramka danych Pandas.
        smiles_col: Nazwa kolumny zawierającej notację SMILES.
        fp_size: Rozmiar wektora fingerprintu.

    Returns:
        pd.DataFrame: Złączona ramka danych zawierająca wejściowe kolumny
        oraz nową macierz cech.
    """
    logging.info(f"Rozpoczynam ekstrakcję cech dla {len(df)} rekordów...")

    # Inicjalizacja rurociągu
    pipeline = build_feature_pipeline(fp_size=fp_size)

    # Wyciągnięcie wektorów cech (X) jako rzadka/gęsta macierz numpy
    X_smiles = df[smiles_col].tolist()
    X_features = pipeline.transform(X_smiles)

    # Konwersja macierzy do ramki danych z odpowiednimi nazwami kolumn
    feature_columns = [f"ECFP_{i}" for i in range(X_features.shape[1])]
    df_features = pd.DataFrame(X_features, columns=feature_columns, index=df.index)

    # Połączenie cech z oryginalną ramką danych
    df_final = pd.concat([df, df_features], axis=1)

    logging.info(f"Ekstrakcja zakończona pomyślnie. Wymiary nowej macierzy: {df_final.shape}")
    return df_final


C:\Users\domin\PycharmProjects\ml-train-lgbm\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
path = "../../1_ontology/data/chebi_dataset_train.parquet"
df = pd.read_parquet(path)
df_preprocessed = create_engineered_dataset(df)

INFO: Rozpoczynam ekstrakcję cech dla 33668 rekordów...
[18:40:32] Running LargestFragmentChooser
[18:40:32] Running Uncharger
[18:40:32] Running LargestFragmentChooser
[18:40:32] Running Uncharger
[18:40:32] Running LargestFragmentChooser
[18:40:32] Running Uncharger
[18:40:32] Running LargestFragmentChooser
[18:40:32] Running Uncharger
[18:40:32] Running LargestFragmentChooser
[18:40:32] Running Uncharger
[18:40:32] Running LargestFragmentChooser
[18:40:32] Running Uncharger
[18:40:32] Removed negative charge.
[18:40:32] Removed negative charge.
[18:40:32] Running LargestFragmentChooser
[18:40:32] Running Uncharger
[18:40:32] Running LargestFragmentChooser
[18:40:32] Running Uncharger
[18:40:32] Running LargestFragmentChooser
[18:40:32] Running Uncharger
[18:40:32] Running LargestFragmentChooser
[18:40:32] Running Uncharger
[18:40:32] Running LargestFragmentChooser
[18:40:32] Running Uncharger
[18:40:32] Running LargestFragmentChooser
[18:40:32] Running Uncharger
[18:40:32] Running L

In [5]:
class ScaffoldSplitter:
    """Realizuje rygorystyczny podział zbioru danych w oparciu o rdzenie Bemis-Murcko.

    Gwarantuje, że cząsteczki o tym samym szkielecie węglowym (scaffold) nie
    znajdą się jednocześnie w zbiorze treningowym i testowym, co pozwala na
    rzetelną ocenę generalizacji modelu.
    """

    @staticmethod
    def get_scaffold(smiles: str) -> str:
        """Generuje szkielet Murcko dla podanego ciągu SMILES.

        Args:
            smiles: Notacja SMILES cząsteczki.

        Returns:
            str: SMILES szkieletu Murcko lub pusty ciąg w przypadku błędu.
        """
        try:
            mol = Chem.MolFromSmiles(smiles)
            if mol:
                # Poprawka: Użycie Chem.MolToSmiles zamiast nieistniejącej metody .ToSmiles()
                scaffold_mol = MurckoScaffold.GetScaffoldForMol(mol)
                return Chem.MolToSmiles(scaffold_mol)
        except Exception as e:
            logging.warning(f"Błąd generowania scaffoldu dla {smiles}: {e}")
        return ""

    def split(
            self,
            df: pd.DataFrame,
            smiles_col: str,
            label_cols: List[str],
            test_size: float = 0.2,
            seed: int = 42
        ) -> Tuple[pd.DataFrame, pd.DataFrame, List[str]]:
            """Dzieli dane na zbiory i usuwa kolumny etykiet bez wariancji.

            Args:
                df: Wejściowa ramka danych.
                smiles_col: Nazwa kolumny zawierającej SMILES.
                label_cols: Lista nazw kolumn z etykietami (np. class_0, class_1...).
                test_size: Proporcja zbioru testowego.
                seed: Ziarno losowości dla powtarzalności.

            Returns:
                Tuple[pd.DataFrame, pd.DataFrame, List[str]]:
                    - df_train: Zbiór treningowy zawierający tylko aktywne klasy.
                    - df_test: Zbiór testowy przefiltrowany do aktywnych klas.
                    - active_labels: Lista nazw klas, które wykazują wariancję w treningu.
            """
            logging.info("Rozpoczynam proces Scaffold Split z filtrowaniem klas stałych...")
            df = df.copy()

            # 1. Generowanie scaffoldów i grupowanie
            df["scaffold"] = df[smiles_col].apply(self.get_scaffold)
            scaffold_sets = df.groupby("scaffold").groups
            unique_scaffolds = list(scaffold_sets.keys())

            # 2. Podział na poziomie unikalnych struktur
            train_scaffolds, test_scaffolds = train_test_split(
                unique_scaffolds,
                test_size=test_size,
                random_state=seed
            )

            train_indices = [idx for s in train_scaffolds for idx in scaffold_sets[s]]
            test_indices = [idx for s in test_scaffolds for idx in scaffold_sets[s]]

            df_train = df.iloc[train_indices].copy()
            df_test = df.iloc[test_indices].copy()

            # 3. Detekcja klas stałych (nunique == 1) w zbiorze TRENINGOWYM
            # To tutaj rozwiązujemy błąd "ValueError: got 1 class"
            y_train = df_train[label_cols]
            trainable_mask = y_train.nunique() > 1
            active_labels = y_train.columns[trainable_mask].tolist()

            dropped_labels = y_train.columns[~trainable_mask].tolist()
            if dropped_labels:
                logging.warning(
                    f"Wykryto i usunięto {len(dropped_labels)} klas stałych (brak wariancji). "
                    f"Przykłady: {dropped_labels[:5]}..."
                )

            # 4. Finalne przygotowanie ramek danych
            # Zachowujemy kolumny pomocnicze (mol_id, SMILES, ECFP_X) oraz tylko AKTYWNE klasy
            non_label_cols = [c for c in df.columns if c not in label_cols and c != "scaffold"]
            final_cols = non_label_cols + active_labels

            df_train_final = df_train[final_cols]
            df_test_final = df_test[final_cols]

            logging.info(
                f"Split zakończony. Pozostało {len(active_labels)} aktywnych klas. "
                f"Train: {len(df_train_final)}, Test: {len(df_test_final)}."
            )

            return df_train_final, df_test_final, active_labels

In [6]:
class HierarchyManager:
    """Zarządza strukturą DAG klasyfikacji ChEBI.

    Odpowiada za wyznaczenie poprawnej kolejności uczenia w łańcuchu (Classifier Chain)
    tak, aby predykcje przodków wspierały predykcje potomków.
    """

    def __init__(self, hierarchy_file: str):
        """Inicjalizuje graf na podstawie pliku definicji klas.

        Args:
            hierarchy_file: Ścieżka do pliku chebi_classes.txt.
        """
        self.graph = nx.DiGraph()
        self._parse_hierarchy(hierarchy_file)

    def _parse_hierarchy(self, path: str) -> None:
        """Parsuje plik tekstowy i buduje graf skierowany."""
        current_id = None
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line.startswith("id:"):
                    current_id = line.split("id: ")[1]
                elif line.startswith("is_a:") and current_id:
                    parent = line.split("is_a: ")[1].split(" ! ")[0]
                    self.graph.add_edge(parent, current_id)

    def get_topological_order(self, available_classes: List[str]) -> List[int]:
        """Oblicza porządek topologiczny dla podzbioru dostępnych klas.

        Args:
            available_classes: Lista nazw kolumn (klas) obecnych w danych.

        Returns:
            List[int]: Indeksy kolumn w optymalnej kolejności dla ClassifierChain.
        """
        # Pobieramy pełny porządek z grafu i filtrujemy do tych, które mamy w df
        full_order = list(nx.topological_sort(self.graph))
        ordered_classes = [c for c in full_order if c in available_classes]

        # Mapujemy nazwy na indeksy pozycji w macierzy Y
        class_to_idx = {name: i for i, name in enumerate(available_classes)}
        return [class_to_idx[c] for c in ordered_classes]

In [7]:
from sklearn.linear_model import SGDClassifier


class TCCModelFactory:
    """Buduje model Topological Classifier Chain z estymatorem LightGBM."""

    @staticmethod
    def create_model(order: List[int]) -> ClassifierChain:
        """Tworzy instancję modelu ClassifierChain z bazowym LightGBM.

        Wykorzystuje parametry zoptymalizowane pod wysoką wymiarowość ECFP.

        Args:
            order: Indeksy kolumn określające kolejność w łańcuchu.

        Returns:
            ClassifierChain: Skonfigurowany model wieloetykietowy.
        """
        # base_lr = LGBMClassifier(
        #     n_estimators=500,
        #     learning_rate=0.05,
        #     max_depth=7,
        #     num_leaves=31,
        #     lambda_l1=0.1,  # Regularyzacja dla rzadkich cech ECFP
        #     lambda_l2=0.1,
        #     class_weight="balanced",
        #     n_jobs=-1,
        #     importance_type="gain",
        #     random_state=42
        # )

        # base_lr = LinearRegression()

        base_lr = SGDClassifier(
            loss="log_loss",          # Wyjście prawdopodobieństwa (wymagane przez Chain)
            penalty="l2",             # Regularyzacja zapobiegająca overfittingowi na ECFP
            alpha=0.0001,             # Siła regularyzacji
            max_iter=1000,            # Szybka zbieżność
            tol=1e-3,
            class_weight="balanced",  # Obsługa niezbalansowanych klas ChEBI
            n_jobs=1,                 # Musi być 1, bo ClassifierChain i tak działa sekwencyjnie
            random_state=42
        )

        return ClassifierChain(base_lr, order=order, random_state=42, verbose=True)

In [9]:
# 1. Split i filtrowanie
all_labels = [c for c in df_preprocessed.columns if c.startswith("class_")]

splitter = ScaffoldSplitter()

df_train, df_test, active_labels = splitter.split(
    df=df_preprocessed,
    smiles_col="SMILES",
    label_cols=all_labels
)

# 2. Budowa porządku topologicznego tylko dla żywych klas
hm = HierarchyManager("chebi_classes.txt")
topological_order = hm.get_topological_order(active_labels)

# 3. Trening na przefiltrowanych danych
X_train = df_train[[c for c in df_train.columns if c.startswith("ECFP_")]].values
y_train = df_train[active_labels].values

INFO: Rozpoczynam proces Scaffold Split z filtrowaniem klas stałych...
[18:41:32] WARNING: not removing hydrogen atom without neighbors
[18:41:32] WARNING: not removing hydrogen atom without neighbors
[18:41:32] WARNING: not removing hydrogen atom without neighbors
[18:41:32] WARNING: not removing hydrogen atom without neighbors
[18:41:32] WARNING: not removing hydrogen atom without neighbors
[18:41:32] WARNING: not removing hydrogen atom without neighbors
[18:41:32] WARNING: not removing hydrogen atom without neighbors
[18:41:32] WARNING: not removing hydrogen atom without neighbors
[18:41:32] WARNING: not removing hydrogen atom without neighbors
[18:41:32] Unusual charge on atom 0 number of radical electrons set to zero
[18:41:33] WARNING: not removing hydrogen atom without neighbors
[18:41:33] WARNING: not removing hydrogen atom without neighbors
[18:41:33] WARNING: not removing hydrogen atom without neighbors
[18:41:33] WARNING: not removing hydrogen atom without neighbors
[18:41:3

In [10]:
# 3. Trening modelu
factory = TCCModelFactory()
model = factory.create_model(order=topological_order)

logging.info("Rozpoczynam trening Topological Classifier Chain...")
model.fit(X_train, y_train)
logging.info("Model został wytrenowany.")

INFO: Rozpoczynam trening Topological Classifier Chain...


[Chain] ................. (1 of 499) Processing order 0, total=   3.4s
[Chain] ............... (2 of 499) Processing order 365, total=   1.1s
[Chain] ............... (3 of 499) Processing order 473, total=   2.4s
[Chain] ............... (4 of 499) Processing order 155, total=   1.8s
[Chain] ............... (5 of 499) Processing order 178, total=   4.8s
[Chain] ................. (6 of 499) Processing order 1, total=   2.1s
[Chain] ................ (7 of 499) Processing order 31, total=   7.4s
[Chain] ............... (8 of 499) Processing order 345, total=   4.5s
[Chain] ............... (9 of 499) Processing order 436, total=   7.4s
[Chain] ................ (10 of 499) Processing order 5, total=   4.3s
[Chain] .............. (11 of 499) Processing order 480, total=   5.5s
[Chain] .............. (12 of 499) Processing order 187, total=   1.6s
[Chain] ............... (13 of 499) Processing order 18, total=   6.5s
[Chain] ................ (14 of 499) Processing order 2, total=   3.1s
[Chain

INFO: Model został wytrenowany.


[Chain] ............. (499 of 499) Processing order 485, total=   1.8s


In [39]:
from sklearn.metrics import f1_score

class ChEBIEvaluator:
    """Realizuje ocenę modelu zgodnie z metrykami Macro F1 oraz Inconsistency.

    Klasa wylicza średni wynik F1 dla wszystkich klas oraz identyfikuje naruszenia
    logiki hierarchicznej na poziomie prawdopodobieństw (probabilistic inconsistency).
    """

    def __init__(
        self,
        label_names: List[str],
        hierarchy_edges: List[Tuple[str, str]]
    ) -> None:
        """Inicjalizuje ewaluator danymi o strukturze klas.

        Args:
            label_names: Lista nazw aktywnych etykiet w kolejności kolumn macierzy Y.
            hierarchy_edges: Lista krawędzi grafu (parent, child) z HierarchyManager.
        """
        self.label_names = label_names
        # Filtrujemy krawędzie tylko do tych klas, które mamy w aktualnym zbiorze
        self.active_edges = [
            (p, c) for p, c in hierarchy_edges
            if p in label_names and c in label_names
        ]
        # Mapowanie nazwy klasy na indeks kolumny dla szybkiego dostępu
        self.label_to_idx = {name: i for i, name in enumerate(label_names)}

    def evaluate(
        self,
        y_true: np.ndarray,
        y_proba: np.ndarray,
        threshold: float = 0.05
    ) -> Dict[str, float]:
        """Oblicza kluczowe metryki: Macro F1 oraz średnią liczbę niespójności.

        Args:
            y_true: Macierz rzeczywistych etykiet binarnych.
            y_proba: Macierz prawdopodobieństw zwrócona przez model (predict_proba).
            threshold: Próg odcięcia dla klasyfikacji binarnej (domyślnie 0.5).

        Returns:
            Dict: Słownik zawierający 'macro_f1' oraz 'avg_inconsistencies'.
        """
        # 1. Obliczenie Macro F1
        y_pred = (y_proba >= threshold).astype(int)
        macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)

        # 2. Obliczenie Inconsistencies (P(child) > P(parent))
        total_inconsistencies = 0
        num_samples = y_proba.shape[0]

        for parent, child in self.active_edges:
            p_idx = self.label_to_idx[parent]
            c_idx = self.label_to_idx[child]

            # Porównanie wektorowe prawdopodobieństw dla wszystkich próbek
            # Inconsistency: P(child) > P(parent)
            inconsistent_mask = y_proba[:, c_idx] > y_proba[:, p_idx]
            total_inconsistencies += np.sum(inconsistent_mask)

        avg_inconsistencies = total_inconsistencies / num_samples

        return {
            "macro_f1": float(macro_f1),
            "avg_inconsistencies": float(avg_inconsistencies),
            "total_samples": float(num_samples)
        }

In [33]:
X_test = df_test[[c for c in df_test.columns if c.startswith("ECFP_")]].values
y_test = df_test[active_labels].values

y_proba_test = model.predict_proba(X_test)

# 2. Inicjalizacja ewaluatora
# hm to Twój HierarchyManager, który posiada graf nx.DiGraph
evaluator = ChEBIEvaluator(
    label_names=active_labels,
    hierarchy_edges=list(hm.graph.edges())
)

# 3. Wyliczenie wyników
results = evaluator.evaluate(y_test, y_proba_test)

logging.info(f"--- RAPORT KOŃCOWY ---")
logging.info(f"Macro F1 Score: {results['macro_f1']:.4f}")
logging.info(f"Average Inconsistencies per Sample: {results['avg_inconsistencies']:.4f}")

if results['avg_inconsistencies'] > 5.0:
    logging.warning("Wysoka liczba niespójności. Rozważ silniejszą regularyzację modelu.")

KeyboardInterrupt: 

In [14]:
from lightgbm import LGBMClassifier
from sklearn.multioutput import MultiOutputClassifier

def build_fast_lgbm_mvp_factory() -> MultiOutputClassifier:
    """Buduje równoległy model LightGBM zoptymalizowany pod kątem szybkości.

    Wykorzystuje MultiOutputClassifier do współbieżnego treningu 500 modeli.
    Parametry LGBM zostały drastycznie odchudzone dla szybkiego MVP,
    ze szczególnym uwzględnieniem rzadkich (sparse) fingerprintów ECFP.

    Returns:
        MultiOutputClassifier: Gotowy do treningu rurociąg.
    """
    base_lgbm = LGBMClassifier(
        # 1. Redukcja złożoności drzewa (Drastyczne przyspieszenie)
        n_estimators=100,        # Zmniejszono z 500 do 100 (wystarczy dla MVP)
        max_depth=5,             # Płytkie drzewa zapobiegają overfittingowi na rzadkich cechach
        num_leaves=15,           # Mniej liści = szybsze budowanie i mniejsze zużycie RAM

        # 2. Subsampling (Przyspieszenie i regularyzacja)
        colsample_bytree=0.3,    # Każde drzewo patrzy tylko na 30% z 2048 cech ECFP!
        subsample=0.8,           # Używa 80% rekordów do budowy pojedynczego drzewa

        # 3. Obsługa rzadkości i niezbalansowania
        class_weight="balanced",
        min_child_samples=10,    # Pozwala na liście dla rzadkich klas

        # 4. Zarządzanie wątkami (KRYTYCZNE)
        # Ponieważ MultiOutputClassifier użyje wszystkich rdzeni,
        # pojedynczy LGBM musi używać tylko 1 wątku, aby uniknąć dławienia CPU.
        n_jobs=1,
        random_state=42,
        verbose=-1               # Wyłączenie spamu w logach z 500 modeli
    )

    # n_jobs=-1 rozdziela trenowanie 500 modeli na wszystkie dostępne rdzenie procesora
    return MultiOutputClassifier(base_lgbm, n_jobs=-1)

In [28]:
logging.info("Rozpoczynam zrównoleglony trening MVP (LightGBM)...")
lgbm_model = build_fast_lgbm_mvp_factory()


lgbm_model.fit(X_train, y_train)
logging.info("Trening LightGBM zakończony.")

INFO: Rozpoczynam zrównoleglony trening MVP (LightGBM)...
INFO: Trening LightGBM zakończony.


In [29]:
class HierarchicalRefiner:
    """Post-processor gwarantujący spójność hierarchiczną predykcji.

    Wymusza zasadę Top-Down Monotonicity, dzięki której prawdopodobieństwo
    przypisania do klasy potomnej (np. kwas tłuszczowy) nigdy nie przekracza
    prawdopodobieństwa przypisania do klasy nadrzędnej (np. lipid).
    """

    def __init__(self, label_names: List[str], hierarchy_edges: List[Tuple[str, str]]):
        """Inicjalizuje refiner na podstawie grafu hierarchii.

        Args:
            label_names: Lista nazw aktywnych klas (kolumn).
            hierarchy_edges: Lista krawędzi grafu w formacie (rodzic, dziecko).
        """
        self.label_names = label_names
        self.label_to_idx = {name: i for i, name in enumerate(label_names)}

        # Filtrujemy krawędzie tylko do tych klas, które realnie trenujemy
        self.hierarchy_edges = [
            (p, c) for p, c in hierarchy_edges
            if p in label_names and c in label_names
        ]

    def refine_probabilities(self, y_proba: np.ndarray) -> np.ndarray:
        """Koryguje macierz prawdopodobieństw, usuwając niespójności logiczne.

        Args:
            y_proba: Surowa macierz prawdopodobieństw [n_samples, n_classes].

        Returns:
            np.ndarray: Skorygowana macierz, w której P(child) <= P(parent).
        """
        y_refined = y_proba.copy()

        # Iteracja po krawędziach grafu. Złożoność to tylko O(E * N).
        for parent, child in self.hierarchy_edges:
            p_idx = self.label_to_idx[parent]
            c_idx = self.label_to_idx[child]

            # Wymuszenie logiczne: P(child) nie może być większe niż P(parent)
            y_refined[:, c_idx] = np.minimum(y_refined[:, c_idx], y_refined[:, p_idx])

        return y_refined

In [31]:
X_test

array([[0, 0, 1, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 2, 0, ..., 0, 0, 0]], dtype=uint32)

In [40]:


# 1. Pobranie surowych predykcji
logging.info("Generowanie surowych prawdopodobieństw...")
y_proba_raw_list = lgbm_model.predict_proba(X_test)

# MultiOutputClassifier dla LGBM zwraca listę tablic. Konwertujemy to na płaską macierz:
y_proba_raw = np.column_stack([p[:, 1] for p in y_proba_raw_list])

# 2. Inicjalizacja Refinera i korekta hierarchii (Rozwiązanie Twojego błędu)
logging.info("Aplikowanie Hierarchical Refiner...")
refiner = HierarchicalRefiner(
    label_names=active_labels,
    hierarchy_edges=list(hm.graph.edges())
)
y_proba_refined = refiner.refine_probabilities(y_proba_raw)

# 3. Ewaluacja finalna
logging.info("Obliczanie finalnych metryk...")
final_results = evaluator.evaluate(y_test, y_proba_refined)

logging.info(f"--- WYNIKI MVP LIGHTGBM ---")
logging.info(f"Macro F1 Score: {final_results['macro_f1']:.4f}")
logging.info(f"Average Inconsistencies per Sample: {final_results['avg_inconsistencies']:.4f}")

INFO: Generowanie surowych prawdopodobieństw...
C:\Users\domin\PycharmProjects\ml-train-lgbm\.venv\lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\Users\domin\PycharmProjects\ml-train-lgbm\.venv\lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\Users\domin\PycharmProjects\ml-train-lgbm\.venv\lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\Users\domin\PycharmProjects\ml-train-lgbm\.venv\lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\Users\domin\PycharmProjects\ml-train-lgbm\.venv\lib\s